In [52]:
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "C:\\Users\\medha\\Downloads\\end to end RAG sy\\data\\goldens.csv"
df.to_csv(csv_path, index=False)

In [54]:
from langsmith import Client

client = Client()
dataset_name = "AgenticAIReportGoldens"

# Store
if client.has_dataset(dataset_name=dataset_name):
    dataset = client.read_dataset(dataset_name=dataset_name)
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Input and expected output pairs for AgenticAIReport",
    )
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)


{'example_ids': ['47325823-564c-4f66-802b-96a91f864d08',
  '59297495-f2ae-4fd4-b5b4-53c566e85ba0',
  'd7cf8b6d-ef7b-4620-afc3-5822f40bc792'],
 'count': 3,
 'as_of': '2026-06-02T05:56:11.181501276Z'}

In [55]:
import sys
sys.path.append(r"C:\Users\medha\Downloads\end to end RAG sy")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "/Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/The 2025 AI Engineering Report.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [56]:
# Test the function with a sample question
test_input = {"question": "For customer-facing applications, which company's models dominate the top rankings?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

Question: For customer-facing applications, which company's models dominate the top rankings?

Answer: Data file not found: /Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/The 2025 AI Engineering Report.txt


In [57]:
from langsmith.evaluation import evaluate, StringEvaluator


In [58]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

Testing all questions from the dataset:

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: Data file not found: /Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/The 2025 AI Engineering Report.txt

--------------------------------------------------------------------------------

Q2: What percentage of respondents are using RAG in some form?
A2: Data file not found: /Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/The 2025 AI Engineering Report.txt

--------------------------------------------------------------------------------

Q3: How often are most respondents updating their models?
A3: Data file not found: /Users/yashpatil/Developer/AI/YT/Sunny/LLMOps_series/data/The 2025 AI Engineering Report.txt

--------------------------------------------------------------------------------



In [59]:
from langsmith.evaluation import evaluate, StringEvaluator
from typing import Optional, Callable

def simple_grading(input_text: str, prediction: str, answer: Optional[str] = None) -> dict:
    """Basic grader: exact-match (case-insensitive) with simple reasoning."""
    try:
        if answer is None:
            score = 0
            reason = "no reference answer"
        else:
            score = 1 if prediction.strip().lower() == answer.strip().lower() else 0
            reason = "exact match" if score == 1 else "mismatch"
    except Exception as e:
        score = 0
        reason = f"error: {e}"
    return {"score": score, "reasoning": reason}

# Evaluators
qa_evaluator = [
    StringEvaluator(
        evaluation_name="cot_qa",
        grading_function=simple_grading,
        input_key="input",
        prediction_key="output",
        answer_key="output",
    )
]
dataset_name = "AgenticAIReportGoldens"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=qa_evaluator,
    experiment_prefix="test-agenticAIReport-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)


View the evaluation results for experiment: 'test-agenticAIReport-qa-rag-d7910cf8' at:
https://smith.langchain.com/o/9162548a-2eb0-46b2-9f4f-67dcd753fba0/datasets/87c1af8e-6667-4133-a333-fbe17aafa203/compare?selectedSessions=4ffd3575-dacf-4728-a6e3-a0e06c40e2bf




0it [00:00, ?it/s]Error running evaluator StringEvaluator(evaluation_name='cot_qa', input_key='input', prediction_key='output', answer_key='output', grading_function=<function simple_grading at 0x000001F8312B4220>) on run 019e86e7-65ed-72e0-8937-58c90315cbf8: KeyError('input')
Traceback (most recent call last):
  File "c:\Users\medha\Downloads\end to end RAG sy\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 1637, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
        run=run,
        example=example,
        evaluator_run_id=evaluator_run_id,
    )
  File "c:\Users\medha\Downloads\end to end RAG sy\.venv\Lib\site-packages\langsmith\evaluation\string_evaluator.py", line 44, in evaluate_run
    run_input = run.inputs[self.input_key]
                ~~~~~~~~~~^^^^^^^^^^^^^^^^
KeyError: 'input'
Error running evaluator StringEvaluator(evaluation_name='cot_qa', input_key='input', prediction_key='output', answer_key='output', grad

In [60]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'reasoning'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "reasoning": reasoning,
            "comment": f"Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "reasoning": f"Error during evaluation: {str(e)}"
        }

In [61]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "AgenticAIReportGoldens"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")

View the evaluation results for experiment: 'agenticAIReport-correctness-eval-8b2ba85e' at:
https://smith.langchain.com/o/9162548a-2eb0-46b2-9f4f-67dcd753fba0/datasets/87c1af8e-6667-4133-a333-fbe17aafa203/compare?selectedSessions=698852a3-86e8-4135-9c88-cafed54720d0




0it [00:00, ?it/s]Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
Please retry in 45.534272115s. [links {
  description: "Lea


Evaluation completed! Check the LangSmith UI for detailed results.


In [62]:
from langsmith.evaluation import evaluate, StringEvaluator

# Combine custom and built-in evaluators
combined_evaluators = [
    correctness_evaluator,  # Custom LLM-as-a-Judge
    StringEvaluator(evaluation_name="cot_qa", grading_function=simple_grading),  # Chain-of-thought QA evaluator
]
